# Stop Putting DBs in Columns

A common anti-pattern in Pandas is nesting dictionaries or lists inside a DataFrame column. This makes operations slow, memory-intensive, and hard to query. In this notebook, we'll see the problem and how to cleanly fix it.

In [ ]:
# Required installation for fresh Colab instances
!pip install pandas

## The Problem: JSON/Dictionaries in a Column

Let's create a DataFrame where the `metadata` column stores a dictionary for every single row. Data Engineers often encounter this when unpacking API responses.

In [ ]:
import pandas as pd
import time

# Generate a DataFrame with a nested dictionary
data = []
for i in range(1, 10001):
    data.append({
        'order_id': i,
        'customer_id': f'C{i%100}',
        'metadata': {'platform': 'web' if i%2==0 else 'mobile', 'version': '1.2.4', 'is_returning': bool(i%3==0)}
    })

df_bad = pd.DataFrame(data)
print("Dataframe with nested JSON dictionary:")
display(df_bad.head())

# Trying to filter or aggregate on the nested data is clunky and slow!
start_time = time.time()

# Example: Count orders from mobile returning users
slow_count = len(df_bad[df_bad['metadata'].apply(lambda x: x.get('platform') == 'mobile' and x.get('is_returning') == True)])
print(f"\nSlow count: {slow_count} (Took {time.time() - start_time:.4f} seconds)")

## The Fix: Flattening the Data

Instead of storing nested structures or using `.apply()`, we should normalize the JSON into its own dedicated columns using `pd.json_normalize()`. This maps everything back into flat memory structures.

In [ ]:
# The Right Way: Normalize the nested dictionaries into standalone columns
df_meta = pd.json_normalize(df_bad['metadata'])

# Combine with the original dataframe (dropping the bad column)
df_good = pd.concat([df_bad.drop(columns=['metadata']), df_meta], axis=1)
print("Normalized Dataframe:")
display(df_good.head())

# Now filtering is extremely fast and uses vectorized boolean comparisons
start_time = time.time()
fast_count = len(df_good[(df_good['platform'] == 'mobile') & (df_good['is_returning'] == True)])
print(f"\nFast count: {fast_count} (Took {time.time() - start_time:.4f} seconds)")

print("\nVectorized operations are significantly faster, safer, and consume less memory than .apply() over nested dicts!")